# 03 — Baseline Training

**Goal:** Train a minimal semantic segmentation CNN from scratch to verify the full pipeline runs end-to-end.

---

## What model are we using?

This notebook uses a **very simple 3-layer CNN** as a placeholder.  
Its purpose is to confirm that:
- Data loading works correctly.
- Tensor shapes are correct at every step.
- The loss goes down (even slightly) after one epoch.
- Checkpointing saves and restores weights.

This model will **not** produce good segmentations — that is expected.  
Once the pipeline is verified, replace the model with U-Net, FCN, or DeepLabv3.

> **Prerequisite:** Run notebooks 00–02 first and ensure `pairs` is populated.

In [1]:
import hashlib
import os
import platform
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data_paths import ARTIFACTS_DIR, MODELS_DIR, PROJECT_ROOT, RAW_DATA_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, load_pairs_from_manifest
from src.foundation import (
    build_checkpoint_metadata,
    detect_dataset_root,
    sha256_file,
    write_run_record,
)
from src.train_utils import get_device, save_checkpoint, train_one_epoch, evaluate

ensure_project_dirs()

print(f"PyTorch version: {torch.__version__}")

All project directories are ready.
PyTorch version: 2.12.1+cu126


## Step 1 — Load Pairs

In [2]:
DATASET_ROOT = detect_dataset_root(RAW_DATA_DIR)
MANIFEST_DIR = ARTIFACTS_DIR / "manifests"
SPLIT_DIR = MANIFEST_DIR / "splits"
DATASET_MANIFEST = MANIFEST_DIR / "ai4mars_dataset_manifest.csv"
SPLIT_HASH_MANIFEST = SPLIT_DIR / "split_manifest_hashes_nav.json"

if not DATASET_MANIFEST.exists():
    raise FileNotFoundError(
        f"Canonical dataset manifest missing: {DATASET_MANIFEST}. "
        "Generate/restore artifacts/manifests before running training."
    )

expected_split_paths = {
    "train": SPLIT_DIR / "train_nav.csv",
    "val": SPLIT_DIR / "val_nav.csv",
    "test_min1_100agree": SPLIT_DIR / "test_min1_100agree_nav.csv",
    "test_min2_100agree": SPLIT_DIR / "test_min2_100agree_nav.csv",
    "test_min3_100agree": SPLIT_DIR / "test_min3_100agree_nav.csv",
}

missing_split_paths = [path for path in expected_split_paths.values() if not path.exists()]
if missing_split_paths:
    missing_text = ", ".join(str(path) for path in missing_split_paths)
    raise FileNotFoundError(
        "Canonical split manifests missing: "
        f"{missing_text}. Generate/restore artifacts/manifests/splits first."
    )

if not SPLIT_HASH_MANIFEST.exists():
    raise FileNotFoundError(
        f"Canonical split hash manifest missing: {SPLIT_HASH_MANIFEST}. "
        "Generate/restore artifacts/manifests/splits first."
    )

split_manifest_paths = expected_split_paths

print(f"Using canonical dataset manifest: {DATASET_MANIFEST}")
for split_name, split_path in sorted(split_manifest_paths.items()):
    print(f"{split_name:20s} -> {split_path.name}")

Using canonical dataset manifest: C:\Users\Jacob\AI4Mars\artifacts\manifests\ai4mars_dataset_manifest.csv
test_min1_100agree   -> test_min1_100agree_nav.csv
test_min2_100agree   -> test_min2_100agree_nav.csv
test_min3_100agree   -> test_min3_100agree_nav.csv
train                -> train_nav.csv
val                  -> val_nav.csv


## Step 2 — Hyperparameters

In [3]:
IMAGE_SIZE   = (256, 256)   # (width, height) - PIL convention
BATCH_SIZE   = 4
NUM_CLASSES  = 4            # soil, bedrock, sand, big_rock
IGNORE_INDEX = 255          # unlabeled pixels in AI4Mars masks
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 3            # quick quality check for weighted + pretrained baseline

## Step 3 — Create Train/Val Split and DataLoaders

In [4]:
TRAIN_SPLIT_SEED = 42
required_split_names = {
    "train",
    "val",
    "test_min1_100agree",
    "test_min2_100agree",
    "test_min3_100agree",
}
missing_names = sorted(required_split_names - set(split_manifest_paths))
if missing_names:
    raise RuntimeError(
        f"Split manifest map is missing required split keys: {missing_names}. "
        "Expected canonical NAV split files under artifacts/manifests/splits."
    )

train_manifest = split_manifest_paths["train"]
val_manifest = split_manifest_paths["val"]
test_manifest_paths = {
    key: value for key, value in split_manifest_paths.items() if key.startswith("test_")
}

train_pairs = load_pairs_from_manifest(
    train_manifest,
    dataset_root=DATASET_ROOT,
    required_label_scheme="NAV",
    require_shape_match=True,
)
val_pairs = load_pairs_from_manifest(
    val_manifest,
    dataset_root=DATASET_ROOT,
    required_label_scheme="NAV",
    require_shape_match=True,
)
expert_test_pairs = {
    name: load_pairs_from_manifest(
        path,
        dataset_root=DATASET_ROOT,
        required_label_scheme="NAV",
        require_shape_match=True,
    )
    for name, path in sorted(test_manifest_paths.items())
}

if "test_min3_100agree" not in expert_test_pairs:
    raise RuntimeError(
        "Canonical expert split 'test_min3_100agree' missing from split manifests."
    )

test_pairs = expert_test_pairs["test_min3_100agree"]

train_dataset = AI4MarsDataset(train_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)
val_dataset = AI4MarsDataset(val_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)
test_dataset = AI4MarsDataset(test_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")
for split_name, split_path in sorted(split_manifest_paths.items()):
    print(f"{split_name:20s} sha256={sha256_file(split_path)}")

cpu_count = os.cpu_count() or 2
if torch.cuda.is_available():
    NUM_WORKERS = max(2, min(8, cpu_count - 1))
else:
    NUM_WORKERS = min(4, max(1, cpu_count // 2))

PIN_MEMORY = torch.cuda.is_available()

print(f"DataLoader num_workers: {NUM_WORKERS}")
print(f"DataLoader pin_memory : {PIN_MEMORY}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

print("Split strategy: canonical committed manifests grouped by acquisition sequence")
print(f"Split seed   : {TRAIN_SPLIT_SEED}")
print("Expert test variant used for training-time holdout checks: min3-100agree")

Train samples: 28487
Val samples  : 7038
Test samples : 526
test_min1_100agree   sha256=69009a3568938307ae1df3bd2a5fca48d104f1b5422f84301b1e5515cdeaebf3
test_min2_100agree   sha256=29fa6638fd2e064c994b025c7da907c0a574bb8c28c1c4b496fe91bae251d8b0
test_min3_100agree   sha256=fa2e5ff07f3c0fb0b376be9b591c7a5731cfa8448ce7d1147e11487304285de7
train                sha256=e7120cc9db78497f10a20ee74da495a0e159af388bf068a335d9ea68820d6057
val                  sha256=9901757ff25027823c60c87d4144b28b198e5337a6b07797e5a1420e1627e21b
DataLoader num_workers: 7
DataLoader pin_memory : True
Split strategy: canonical committed manifests grouped by acquisition sequence
Split seed   : 42
Expert test variant used for training-time holdout checks: min3-100agree


## Step 3.5 — Compute Data-Driven Class Weights

Estimate class frequencies from the training split so the loss reflects the
actual NAV label distribution instead of an approximate hand-tuned vector.

In [5]:
stats_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

class_pixel_counts = torch.zeros(NUM_CLASSES, dtype=torch.long)
for _, masks in stats_loader:
    valid = masks != IGNORE_INDEX
    for class_id in range(NUM_CLASSES):
        class_pixel_counts[class_id] += ((masks == class_id) & valid).sum()

total_labeled_pixels = class_pixel_counts.sum().item()
class_frequencies = class_pixel_counts.float() / max(total_labeled_pixels, 1)
class_weights = 1.0 / torch.clamp(class_frequencies, min=1e-8)
class_weights = class_weights / class_weights.mean()

print("Train-split labeled pixel counts by class:")
for class_id, class_name in enumerate(["soil", "bedrock", "sand", "big_rock"]):
    print(
        f"  class {class_id} ({class_name:>10s}): {class_pixel_counts[class_id].item():>12d} pixels "
        f"({class_frequencies[class_id].item() * 100:6.2f}%)"
    )

print("\nEmpirical class weights (inverse-frequency, mean-normalized):")
for class_id, class_name in enumerate(["soil", "bedrock", "sand", "big_rock"]):
    print(f"  class {class_id} ({class_name:>10s}): {class_weights[class_id].item():.4f}")

Train-split labeled pixel counts by class:
  class 0 (      soil):    485292480 pixels ( 39.57%)
  class 1 (   bedrock):    348943148 pixels ( 28.45%)
  class 2 (      sand):    336692129 pixels ( 27.45%)
  class 3 (  big_rock):     55489016 pixels (  4.52%)

Empirical class weights (inverse-frequency, mean-normalized):
  class 0 (      soil): 0.3180
  class 1 (   bedrock): 0.4423
  class 2 (      sand): 0.4584
  class 3 (  big_rock): 2.7813


## Step 4 — Verify Batch Shapes

Always check tensor shapes **before** training.  
A shape mismatch will cause a confusing error inside the model.

In [6]:
images, masks = next(iter(train_loader))
print(f"Image batch shape : {images.shape}   (expected [B, 3, H, W])")
print(f"Mask  batch shape : {masks.shape}    (expected [B, H, W])")
print(f"Image dtype       : {images.dtype}   (expected float32)")
print(f"Mask  dtype       : {masks.dtype}    (expected int64 / long)")
print(f"Image value range : [{images.min():.3f}, {images.max():.3f}]  (expected [0, 1])")

Image batch shape : torch.Size([4, 3, 256, 256])   (expected [B, 3, H, W])
Mask  batch shape : torch.Size([4, 256, 256])    (expected [B, H, W])
Image dtype       : torch.float32   (expected float32)
Mask  dtype       : torch.int64    (expected int64 / long)
Image value range : [0.000, 1.000]  (expected [0, 1])


## Step 5 — Define a Stronger Pretrained Segmentation Model

For the first real baseline, use a pretrained U-Net encoder so the model starts with useful low-level visual features (edges, textures) instead of learning everything from scratch.

In [7]:
import segmentation_models_pytorch as smp

# Use a lighter encoder on CPU to keep iterations practical.
device = get_device()
encoder_name = "mobilenet_v2" if device.type == "cpu" else "resnet34"

model = smp.Unet(
    encoder_name=encoder_name,
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

# Quick shape check before training
dummy = torch.randn(2, 3, IMAGE_SIZE[1], IMAGE_SIZE[0]).to(device)
out = model(dummy)
print(f"Encoder: {encoder_name}")
print(f"Model output shape: {out.shape}   (expected [2, {NUM_CLASSES}, {IMAGE_SIZE[1]}, {IMAGE_SIZE[0]}])")

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

Using device: cuda
Encoder: resnet34
Model output shape: torch.Size([2, 4, 256, 256])   (expected [2, 4, 256, 256])
Model parameters: 24,436,804


## Step 6 — Loss Function and Optimiser

In [8]:
# Weighted cross-entropy to counter class imbalance using the empirical class
# distribution computed above.
weights = class_weights.to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=IGNORE_INDEX)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Class weights: {weights.tolist()}")

Class weights: [0.3180195689201355, 0.4422855079174042, 0.4583786725997925, 2.7813162803649902]


## Reproducibility Log

Record the key settings needed to reproduce the baseline experiment and the
saved checkpoint.

In [9]:
PREPROCESSING_CONFIG = {
    "image_size": list(IMAGE_SIZE),
    "require_original_shape_match": True,
    "label_scheme": "NAV",
}
LOSS_NAME = "CrossEntropyLoss"
LOSS_WEIGHTS = [float(value) for value in class_weights.tolist()]
MODEL_NAME = f"Unet/{encoder_name}"
RUN_CONFIG = {
    "model_name": MODEL_NAME,
    "epochs": int(NUM_EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "seed": int(TRAIN_SPLIT_SEED),
    "preprocessing": PREPROCESSING_CONFIG,
    "loss_name": LOSS_NAME,
    "loss_weights": LOSS_WEIGHTS,
}

print("Prepared canonical provenance config for checkpoint + run record:")
print(f"  dataset_manifest: {DATASET_MANIFEST}")
for split_name, split_path in sorted(split_manifest_paths.items()):
    print(f"  {split_name:20s} sha256={sha256_file(split_path)}")
print(f"  model_name      : {MODEL_NAME}")
print(f"  loss_name       : {LOSS_NAME}")
print(f"  loss_weights    : {LOSS_WEIGHTS}")

Prepared canonical provenance config for checkpoint + run record:
  dataset_manifest: C:\Users\Jacob\AI4Mars\artifacts\manifests\ai4mars_dataset_manifest.csv
  test_min1_100agree   sha256=69009a3568938307ae1df3bd2a5fca48d104f1b5422f84301b1e5515cdeaebf3
  test_min2_100agree   sha256=29fa6638fd2e064c994b025c7da907c0a574bb8c28c1c4b496fe91bae251d8b0
  test_min3_100agree   sha256=fa2e5ff07f3c0fb0b376be9b591c7a5731cfa8448ce7d1147e11487304285de7
  train                sha256=e7120cc9db78497f10a20ee74da495a0e159af388bf068a335d9ea68820d6057
  val                  sha256=9901757ff25027823c60c87d4144b28b198e5337a6b07797e5a1420e1627e21b
  model_name      : Unet/resnet34
  loss_name       : CrossEntropyLoss
  loss_weights    : [0.3180195689201355, 0.4422855079174042, 0.4583786725997925, 2.7813162803649902]


## Step 7 — Training Loop

In [10]:
CLASS_NAMES = ["soil", "bedrock", "sand", "big_rock"]

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_metrics = evaluate(
        model,
        val_loader,
        loss_fn,
        device,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX,
        return_per_class_iou=(epoch == NUM_EPOCHS),  # avoid extra overhead each epoch
    )

    print(f"  Train loss : {train_loss:.4f}")
    print(f"  Val loss   : {val_metrics['val_loss']:.4f}")
    print(f"  Pixel acc  : {val_metrics['pixel_acc']:.4f}")
    print(f"  Mean IoU   : {val_metrics['mean_iou']:.4f}")

    per_class_iou = val_metrics.get("per_class_iou")
    if per_class_iou is not None:
        print("  Per-class IoU:")
        valid_scores = []
        for class_name, score in zip(CLASS_NAMES, per_class_iou):
            if score is None:
                print(f"    - {class_name:8s}: n/a")
            else:
                print(f"    - {class_name:8s}: {score:.4f}")
                valid_scores.append((class_name, score))

        if valid_scores:
            hardest_class, hardest_score = min(valid_scores, key=lambda x: x[1])
            print(f"  Hardest class this epoch: {hardest_class} (IoU={hardest_score:.4f})")


Epoch 1/3
----------------------------------------


  Batch 10/7122  loss=1.3085
  Batch 20/7122  loss=1.2526
  Batch 30/7122  loss=1.4708
  Batch 40/7122  loss=1.9911
  Batch 50/7122  loss=1.3457
  Batch 60/7122  loss=1.4920
  Batch 70/7122  loss=1.1017
  Batch 80/7122  loss=1.1525
  Batch 90/7122  loss=1.3187
  Batch 100/7122  loss=1.1227
  Batch 110/7122  loss=1.0403
  Batch 120/7122  loss=1.0500
  Batch 130/7122  loss=1.5627
  Batch 140/7122  loss=0.8548
  Batch 150/7122  loss=1.1020
  Batch 160/7122  loss=1.2309
  Batch 170/7122  loss=1.0106
  Batch 180/7122  loss=1.1638
  Batch 190/7122  loss=2.2201
  Batch 200/7122  loss=1.3369
  Batch 210/7122  loss=0.9429
  Batch 220/7122  loss=1.0858
  Batch 230/7122  loss=1.0551
  Batch 240/7122  loss=0.9993
  Batch 250/7122  loss=1.0561
  Batch 260/7122  loss=1.4527
  Batch 270/7122  loss=1.0245
  Batch 280/7122  loss=1.4961
  Batch 290/7122  loss=1.4601
  Batch 300/7122  loss=1.1017
  Batch 310/7122  loss=1.1746
  Batch 320/7122  loss=1.2953
  Batch 330/7122  loss=1.2917
  Batch 340/7122  l

## Step 8 — Save Checkpoint

In [11]:
checkpoint_metadata = build_checkpoint_metadata(
    project_root=PROJECT_ROOT,
    dataset_manifest_path=DATASET_MANIFEST,
    split_manifest_paths=split_manifest_paths,
    active_split_name="val",
    preprocessing=PREPROCESSING_CONFIG,
    loss_name=LOSS_NAME,
    loss_weights=LOSS_WEIGHTS,
    model_name=MODEL_NAME,
    seed=TRAIN_SPLIT_SEED,
)

checkpoint_path = MODELS_DIR / "weighted_unet_epoch03.pth"
save_checkpoint(
    model,
    optimizer,
    epoch=NUM_EPOCHS,
    path=checkpoint_path,
    metadata=checkpoint_metadata,
)

run_id = f"weighted_unet_epoch{NUM_EPOCHS:02d}"
run_dir = ARTIFACTS_DIR / "runs" / run_id
metrics_payload = {
    "epoch": int(NUM_EPOCHS),
    "train_loss": float(train_loss),
    "val_loss": float(val_metrics["val_loss"]),
    "pixel_acc": float(val_metrics["pixel_acc"]),
    "mean_iou": float(val_metrics["mean_iou"]),
    "finite_loss_batches": int(val_metrics["finite_loss_batches"]),
    "skipped_all_ignore_loss_batches": int(val_metrics["skipped_all_ignore_loss_batches"]),
}
if val_metrics.get("per_class_iou") is not None:
    metrics_payload["per_class_iou"] = [
        None if value is None else float(value)
        for value in val_metrics["per_class_iou"]
    ]

config_payload = RUN_CONFIG
environment_text = "\n".join([
    f"python={sys.version}",
    f"platform={platform.platform()}",
    f"git_commit={checkpoint_metadata['git_commit_sha']}",
    f"requirements_sha256={checkpoint_metadata['dependency_lock_hash']}",
]) + "\n"
write_run_record(
    run_dir,
    config=config_payload,
    dataset_manifest_path=DATASET_MANIFEST,
    split_manifest_paths=split_manifest_paths,
    metrics=metrics_payload,
    environment_text=environment_text,
)
print(f"Saved checkpoint to {checkpoint_path}")
print(f"Wrote lightweight run record to {run_dir}")

Checkpoint saved -> C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth
Saved checkpoint to C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth
Wrote lightweight run record to C:\Users\Jacob\AI4Mars\artifacts\runs\weighted_unet_epoch03


## Next Steps

- Open `04_evaluation_error_analysis.ipynb` to load the checkpoint and analyse predictions.
- Run a cleaner experiment series next:
    - **Baseline:** U-Net with a ResNet34 encoder.
    - **Alternative encoder:** U-Net with EfficientNet if it fits local hardware.
    - **Alternative decoder family:** DeepLabV3+ for stronger context aggregation.
    - **Loss experiments:** Dice, Focal, or CE+Dice hybrids.